# M2P2 - Bash Utilities practice and AWK usage

This Notebook will cover the use of bash utilities for the handling of NGS data for preprocessing, parsing, and file management. It will demonstrate the following: 

- Merging files
- Compressing and modifying file types
- Indexing files
- Counting entries in various file types (`fasta`, `fastq`, `sam`, etc.)
- Usage of `AWK` and `grep` for parsing files




-----

## Loading of the Module 2 environment, `v1.0`
From here on in the course, we will be using enviornments a little bit differently. They still function in the same way, acting as containerized collections of programs and their dependencies, but they will utilize the `Modules` tool. This is an environment manager setup to work specifically within Hellbender's HPC cluster. We will use the `ml`, or module load, command to load our enviornments from here on out. This week's environment is called `v1.0`, and is located within the sym path `class/mendoza/class-bbme4001-7001/v1.0`. This environment contains the following programs: 
| Tool                  | Documentation Link                                                                 |
|-----------------------|------------------------------------------------------------------------------------|
| **FastQC**            | [FastQC Documentation](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/) |
| **fastp**             | [fastp GitHub Repository](https://github.com/OpenGene/fastp)                       |
| **MultiQC**           | [MultiQC Documentation](https://docs.seqera.io/multiqc)                            |
| **Trimmomatic**       | [Trimmomatic Manual V0.32](https://www.usadellab.org/cms/uploads/supplementary/Trimmomatic/TrimmomaticManual_V0.32.pdf) |
| **SRA-Toolkit**       | [SRA-Toolkit Documentation](https://hpc.nih.gov/apps/sratoolkit.html)               |
| **Entrez Direct**     | [Entrez Direct Documentation](https://www.ncbi.nlm.nih.gov/books/NBK179288/)       |
| **ncbi-datasets-cli** | [NCBI Datasets CLI Documentation](https://www.ncbi.nlm.nih.gov/datasets/docs/v2/)   |
| **Samtools**          | [Samtools Documentation](https://www.htslib.org/doc/)                               |
| **BEDTools**          | [BEDTools Documentation](https://bedtools.readthedocs.io/en/latest/)               |
| **Seqtk**             | [Seqtk GitHub Repository](https://github.com/lh3/seqtk)                            |
| **EMBOSS**            | [EMBOSS Documentation](https://emboss.open-bio.org/Apps)                           |
| **BLAST**             | [BLAST+ Documentation](https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/LATEST/) |
| **IGV**               | [IGV Documentation](https://software.broadinstitute.org/software/igv/)            |
| **curl**              | [curl Documentation](https://curl.se/docs/)                                         |
| **pigz**              | [pigz GitHub Repository](https://github.com/madler/pigz)                           |
| **jq**                | [jq Manual](https://stedolan.github.io/jq/manual/)                                  |
| **ipykernel**         | [ipykernel Documentation](https://ipython.readthedocs.io/en/stable/)   

To load this environment, `v1.0`, please use the following command: `ml class/mendoza/class-bbme4001-7001/v1.0`. To see all available modules within the cluster, you can use either `module spider` or `ml spider`. `ml list` will show you which modules are currently active in your session. See `ml --help` for more information on the `Modules` tool. This module also contains ipykernel, which will allow you to run bash commands using the programs within this module. We can also install the `bash_kernel` program in our Jupyter build, then we can natively run bash commands without the use of the bash magic(`%%bash` for scripts or `%` / `!` before lines of bash within the code cells in Jupyter notebook). This will allow us to load the `v1.0` environment with our Jupyter Notebook with ease. 

**Please run the code below, then refresh the page and select the `Bash` kernel within Jupyter notebook.** 

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

In [ ]:
%%bash
ml class/mendoza/class-bbme4001-7001/v1.0
pip install bash_kernel
python -m bash_kernel.install


**At this point, you should have the bash kernel installed in Jupyter and load the `bash` terminal.** 
![image.png](images/M2P2_Bash_Utilities_Practice_and_AWK_f9bffab9.png)

In [ ]:
#Run this to load the v1.0 environment with the Modules program into our bash cells
ml class/mendoza/class-bbme4001-7001/v1.0

In [ ]:
#first, navigate to the proper folder within these cells. Check where you are with pwd
pwd

In [ ]:
#You should be in module_2_ngs_preprocessing/practice/. If you are not, navigate there.
#Inputs are created by: bash data/fetch_data.sh practice
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md) 

----

## Merging and Converting file types

Bash is a useful and simple language; thus, bash tools can be used to manipulate data very quickly. Below are some examples of how to sort NGS data that will come in handy later on in this course: 

#### Merging files
When multiple sequencing lanes or runs correspond to the same sample, the FASTQ files must be **merged** before trimming or alignment.


In [ ]:
# Normally, you’d merge large fastq.gz files. This is just the syntax. We use the cat command to make the two files into a concatenated single file.
#Will result in an error
cat data/raw/Sample1_L00*_R1.fastq.gz > data/raw_merged/Sample1_R1_merged.fastq.gz

#### Compressing Files
We can compress files to `gzip`(`.gz`) format with the `gzip` command. To unzip `.gz` files, we use the command `gunzip`.

In [ ]:
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md)
#check the layout of the fasta file from last week called Potential_Iron_Homeostasis_Genes_Arabidopsis_cDNA.fa
head "$DATA_DIR"/practice/m2p2/Potential_Iron_Homeostasis_Genes_Arabidopsis_cDNA.fa

#check the contents of the $DATA_DIR/practice/m2p2/ directory with ls -l. Note the file size of 103 KB
ls -lh "$DATA_DIR"/practice/m2p2/

In [ ]:
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md)
#now we will convert this file to a gzip format, it's file type will be .fa.gz
gzip "$DATA_DIR"/practice/m2p2/Potential_Iron_Homeostasis_Genes_Arabidopsis_cDNA.fa

#check the contents of the $DATA_DIR/practice/m2p2/ directory with ls after compression - 32 KB, which is a 60% reduction in storage size
ls -lh "$DATA_DIR"/practice/m2p2/

In [ ]:
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md)
#Now we will decompress the file
gunzip "$DATA_DIR"/practice/m2p2/Potential_Iron_Homeostasis_Genes_Arabidopsis_cDNA.fa.gz

#And check again with ls -l
ls -lh "$DATA_DIR"/practice/m2p2/

## Indexing files with samtools

We will use [`samtools`](https://www.htslib.org/doc/) to index files. The example here is indexing an *Arabidopsis thaliana* genome. Indexing allows for rapid access to very large files, and it creates a `.Xai` (`fa.fai` for fasta and `bam.bai` for bam files) file that contains a list of the sequence names and lengths.

In [ ]:
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md)
#index the A. thaliana genome with samtools:
samtools faidx "$DATA_DIR"/practice/m2p2/Arabidopsis_thaliana.TAIR10.cdna.all.fa

#Note the creation of the .fa.fai file from the samtools program
ls "$DATA_DIR"/practice/m2p2/

#check out what the fai looks like: 
echo "head of the Arabidopsis_thaliana.TAIR10.cdna.all.fa.fai"
head "$DATA_DIR"/practice/m2p2/Arabidopsis_thaliana.TAIR10.cdna.all.fa.fai

## `grep` and `awk` for parsing files
These are powerful text-processing tools for extracting information with the bash command line. `grep` commands have been reviewed in Module 1, and we will cover `awk` commands today. 

#### Extracting chromosome 1 from a `fasta` file:

In [ ]:
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md)
#This command will parse the 1st chromosome from the file "$DATA_DIR"/practice/m2p2/Arabidopsis_thaliana.TAIR10.cdna.fa
awk '/^>Chr1/,/^>/{if($0 !~ /^>/) print}' "$DATA_DIR"/practice/m2p2/Arabidopsis_thaliana.TAIR10.cdna.all.fa > "$DATA_DIR"/practice/m2p2/Arabidopsis_thaliana.TAIR10.cdna.Chr1.fa

#### `awk` Command Examples:
The `awk` utility is a text-processing language commonly used in bioinformatics for filtering, parsing, and summarizing data from files. We will generally use it on FASTA, FASTQ, SAM, and GTF formats.

| Command | Description | Example Input File | Example Output / Purpose |
|----------|--------------|--------------------|---------------------------|
| `awk '{print $1}' file.txt` | Prints the **first column** of each line | Tab-delimited table | Extract gene IDs from a count matrix |
| `awk '{print $1, $3}' file.txt` | Prints **column 1 and 3** separated by space | Any text file | Extracts specific fields |
| `awk -F "\t" '{print $2}' file.txt` | Uses **tab as a field separator** | TSV file | Extract 2nd column when fields are tab-separated |
| `awk '/pattern/' file.txt` | Prints lines that **match a pattern** | FASTA, GTF, etc. | Find lines containing “AT1G” |
| `awk '!/pattern/' file.txt` | Prints lines that **do not** match a pattern | Any text file | Exclude unwanted entries |
| `awk '{if($3 > 1000) print $0}' file.txt` | Conditional filtering | Numeric tables | Print rows where column 3 > 1000 |
| `awk 'NR==1 {print $0}' file.txt` | Print **first line only** | Any file | Extract header line |
| `awk 'NR>1 {print $1}' file.txt` | Skip header and print column 1 | Any table | Get sample names excluding header |
| `awk '{sum += $3} END {print sum}' file.txt` | Sum all values in column 3 | Count table | Total number of mapped reads |
| `awk '{count++} END {print count}' file.txt` | Count number of lines | FASTA, FASTQ, SAM | Total lines in file |
| `awk '{if(NR%4==1) print $0}' reads.fastq` | Print every 4th line starting at 1 | FASTQ | Extract read IDs |
| `awk '{if(NR%4==2) print length($0)}' reads.fastq` | Print sequence lengths | FASTQ | Check read lengths |
| `awk '{print NF}' file.txt` | Prints **number of fields** per line | Any delimited text | Diagnose inconsistent columns |
| `awk -F"\t" '{OFS="\t"; print $1, toupper($2)}' file.txt` | Modify case (to upper) | Any text | Transform text within fields |
| `awk '{if($1 ~ /^#/ ) next; print $1}' file.gtf` | Skip comment lines | GTF / GFF | Filter metadata lines starting with `#` |
| `awk 'BEGIN{FS="\t"; OFS="\t"} {print $1,$2,$3}' file.txt` | Set input/output field separators | TSV | Reformat data while preserving tabs |


In [ ]:
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md)
#Here we use awk to print reads in our fastq file, small_SRR1971253.fastq
awk 'END {print NR/4 " reads in total"}' "$DATA_DIR"/practice/m2p2/small_SRR1971253.fastq

We can even use awk to parse files for specific genes, such as searching the genome cDNA from *A. Thaliana* as we did in M1E1. 

In [ ]:
: "${DATA_DIR:=../../../data}"   # repository data/ folder (see data/README.md)
awk '/^>.*AT1G76820.1/{flag=1; print; next} /^>/{flag=0} flag{print}' "$DATA_DIR"/practice/m2p2/Arabidopsis_thaliana.TAIR10.cdna.all.fa

---
### Try it on your own: 
Can you think of a way to use a for loop to parse the `Arabidopsis_thaliana.TAIR10.cdna.all.fa` file for the genes within the file `Potential_Iron_Homeostasis_Genes_Arabidopsis.txt` as we had before, but this time using awk rather than the `blast` program within the terminal?

In [ ]:
#Use awk to search the A. Thaliana cDNA file for the 48 genes in Potential_Iron_Homeostasis_Genes_Arabidopsis.txt to generate a file called AWK Potential_Iron_Homeostasis_Genes_Arabidopsis_cDNA.fa

### Resource links for `grep`, `awk`, and learning bash utilities:
| Tool          | Documentation / Learning Link                                                      |
|---------------------------|-----------------------------------------------------------------------------------|
| **grep**                  | [GNU grep Manual](https://www.gnu.org/software/grep/manual/grep.html)             |
| **awk**                   | [GNU awk User’s Guide](https://www.gnu.org/software/gawk/manual/gawk.html)       |
| **Bash Reference Manual** | [GNU Bash Manual](https://www.gnu.org/software/bash/manual/bash.html)            |
| **Bash Beginner Tutorials** | [GNU Bash Introduction](https://www.gnu.org/software/bash/manual/bash.html#Introduction) |
| **Bash Scripting Guide**  | [Advanced Bash-Scripting Guide](https://tldp.org/LDP/abs/html/)                  |
| **Bash One-Liners**       | [Bash One-Liners Explained](https://www.bash.academy/bash-one-liners/)           |
| **Regular Expressions (grep/awk)** | [Regular Expressions Tutorial](https://www.regular-expressions.info/)       |